FashionMNIST

简单的多层感知机（MLP）；（核心结构）<br>
输入➡️展平➡️全连接+ReLU➡️隐藏层➡️输出层<br>
注意：我用的是macbook，无CUDA加速<br>

导入数据库

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"当前使用的设备: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) #均值，标准差
])

train_set = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True) #顺序打乱

print(f"成功加载 {len(train_set)} 张图片！")

当前使用的设备: mps<br>
成功加载 60000 张图片！<be>

来来来，先解释一下这些库的作用<br>
torch：张量计算，自动微分<br>
torch.nn:神经网络层（卷积，全连接等）损失函数<br>
torch.optim:优化器（SGD，Adam等）<br>
torchvision.datasets:数据集的下载加载<br>
torchvision.transforms:对于图像进行预处理（转tensor，归一化）<br>
torch.utils.data.DataLoader:数据集包装成迭代加载器<br>

如果是GPU（cuda）<br>
修改代码<br>
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")<br>
print(f"当前使用的设备: {device}")<br>

transforms.Compose:执行一下操作<br>
当前不执行，当数据集输入时，立刻执行，但代码中不显示。<br>
transforms.PyTorch：0-255变成0-1（浮点数），形状➡️通道，高，宽<br>
transforms.Normalize:归一化，标准化，（原值-均值）/标准差，[0,1]➡️[-1,1]<br>

download = True是指当文件夹里没有的数据进行下载，有的跳过

$$ normalize = \frac{x - {mean}}{SD} $$

数据的一些小看一下

In [ ]:
import matplotlib.pyplot as plt

labels_map = {
    0: "T-Shirt", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
    5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle Boot"
}

figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(train_set), size=(1,)).item()
    img, label = train_set[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

labels_map：一一对应<br>
figure = plt.figure(figsize=(8, 8))这里的$8*8$是整个图的大小，如果改成（10， 10）是$10*10$会更大<br>
cols, rows = 3, 3这里是$3*3$的图片,可以看一下image里的$5*5$的<br>
len(train_set) = 60000，上面有<br>
torch.randint(上限， 张量形状)，.item()取普通整数，size = (1,)一维tensor一个数，（a个数，b维），a行*b列<br>
add_subplot是指开小窗口，比如一个大图上有3*3的小图，取其一填充<br>
i是从左到右，从上到下的开始<br>
plt.axis("off")关闭坐标轴，刻度，边框<br>
plt.imshow(img.squeeze(), cmap="gray"):一张图输出，squeeze去除维度中的1，使其变成28*28,cmap是颜色映射，gray灰色<br>
plt.imshow后台呈现，plt.show前台呈现<br>

In [7]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.flatten = nn.Flatten() # 把 28x28 展平为 784
        self.linear_relu_stack = nn.Sequential(  #按顺序执行
            nn.Linear(28*28, 512),   # 输入层
            nn.ReLU(),               # 激活函数(正数保留，负数为0)
            nn.Linear(512, 256),     # 隐藏层
            nn.ReLU(),
            nn.Linear(256, 10)       # 输出层（10个类别）
        )

    def forward(self, x):
        x = self.flatten(x) #flatten自动去除1，x本身为（1，28，28）
        logits = self.linear_relu_stack(x)
        return logits

model = SimpleMLP().to(device)  #用CPU还是GPU跑
print(model)

SimpleMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


SimpleMLP(<br>
  (flatten): Flatten(start_dim=1, end_dim=-1)<br>
  (linear_relu_stack): Sequential(<br>
    (0): Linear(in_features=784, out_features=512, bias=True)<br>
    (1): ReLU()<br>
    (2): Linear(in_features=512, out_features=256, bias=True)<br>
    (3): ReLU()<br>
    (4): Linear(in_features=256, out_features=10, bias=True)<br>
  )<br>
)<br>

定义SimpleMLP模型，从nn.Module继承<br>
super().__init__()重新定义模型，__init__相当于激活pytorch的一个引子<br>
forwand：将输入变成输出<br>

一层隐藏层变多层隐藏层？：<nb>
可以但没有必要，模型数据本身比较简单，不需要进行多层的隐藏层，多层的话会过拟合
512，256变成其他数？：<br>
可以，一般以$2^n$为基准进行数据填充，一般来说数据大小逐步减少

In [ ]:
loss_fn = nn.CrossEntropyLoss() #交叉熵损失函数（内涵Softmax激活函数，不需要另外加）
optimizer = optim.Adam(model.parameters(), lr=1e-3) #Adam优化器

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 计算预测误差
        pred = model(X)
        loss = loss_fn(pred, y)

        # 反向传播
        optimizer.zero_grad() #清空上一轮梯度
        loss.backward() #反向计算梯度
        optimizer.step() #更新模型参数

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
print("训练完成！")

lr：学习率，1e-3=0.001

我用的是MacBook，如果是多张的GPU呢，那么代码该怎么写对于模型训练最好（这个模型和数据很简单，不需要多张）<br>
device = torch.device("cuda":0 if torch.cuda.is_available() else "cpu")<br>
if torch.cuda.device_count()>1:<br>
    model = nn.DataParallel(model)<br>
    model = model.to(device)<br>

Epoch 1<br>
-------------------------------<br>
loss: 2.278525  [    0/60000]<br>
loss: 0.433730  [ 6400/60000]<br>
loss: 0.408205  [12800/60000]<br>
loss: 0.644453  [19200/60000]<br>
loss: 0.438606  [25600/60000]<br>
loss: 0.523119  [32000/60000]<br>
loss: 0.390015  [38400/60000]<br>
loss: 0.473486  [44800/60000]<br>
loss: 0.369046  [51200/60000]<br>
loss: 0.385244  [57600/60000]<br>
Epoch 2<br>
-------------------------------<br>
loss: 0.315299  [    0/60000]<br>
loss: 0.417129  [ 6400/60000]<br>
loss: 0.341317  [12800/60000]<br>
loss: 0.491811  [19200/60000]<br>
loss: 0.288891  [25600/60000]<br>
loss: 0.360915  [32000/60000]<br>
loss: 0.333595  [38400/60000]<br>
loss: 0.228548  [44800/60000]<br>
loss: 0.387704  [51200/60000]<br>
loss: 0.578608  [57600/60000]<br>
Epoch 3<br>
-------------------------------<br>
loss: 0.267518  [    0/60000]<br>
loss: 0.420243  [ 6400/60000]<br>
loss: 0.319905  [12800/60000]<br>
loss: 0.356769  [19200/60000]<br>
loss: 0.557706  [25600/60000]<br>
loss: 0.231297  [32000/60000]<br>
loss: 0.311977  [38400/60000]<br>
loss: 0.433480  [44800/60000]<br>
loss: 0.221533  [51200/60000]<br>
loss: 0.316420  [57600/60000]<br>
Epoch 4<br>
-------------------------------<br>
loss: 0.321993  [    0/60000]<br>
loss: 0.283508  [ 6400/60000]<br>
loss: 0.334996  [12800/60000]<br>
loss: 0.331831  [19200/60000]<br>
loss: 0.288601  [25600/60000]<br>
loss: 0.485527  [32000/60000]<br>
loss: 0.204387  [38400/60000]<br>
loss: 0.289070  [44800/60000]<br>
loss: 0.293278  [51200/60000]<br>
loss: 0.306726  [57600/60000]<br>
Epoch 5<br>
-------------------------------<br>
loss: 0.287756  [    0/60000]<br>
loss: 0.156889  [ 6400/60000]<br>
loss: 0.216232  [12800/60000]<br>
loss: 0.266839  [19200/60000]<br>
loss: 0.322872  [25600/60000]<br>
loss: 0.199405  [32000/60000]<br>
loss: 0.158567  [38400/60000]<br>
loss: 0.322434  [44800/60000]<br>
loss: 0.307890  [51200/60000]<br>
loss: 0.246459  [57600/60000]<br>
训练完成！<br>

Epoch 1模型的loss从2.27迅速下降到0.38，模型快速学到了基本规律<br>
Epoch 2模型初步探索，loss回升至0.57算是正常<nr>
Epoch 3模型逐步趋于稳定，loss变化幅度减少<br>
Epoch 4模型的loss稳定在0.2-0.3之间了<br>
Epoch 5模型的loss下降至local min，训练结束<br>

当Epoch = 10时，5后会有几率loss到0.11，0.13等，但是最后loss回升至0.24，有过拟合的风险
原因：
    1.可能是lr过大，把后续的lr变成1e-4（原来的1/10）
    2.数据过于简单，那个loss为0.11，0.13可能由于数据过于简单（过于雷同），导致loss出现奇迹般的历史新低
    3.过拟合的前兆

修改代码：
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1) 

for t in range(epochs):
    train_loop(train_loader, model, loss_fn, optimizer)
    test_loop(test_loader, model, loss_fn)
    scheduler.step()